## What are recommender systems?

A recommender system is an information filtering tool used to predict a user's rating.

Example applications: 
1. E-commerce websites (recommending a product to buy based on previous purchase and "similar" buyer's further purchase items- Amazon),<br>
2. Streaming services that suggest movies/music (Spotify, Netflix)
3. Social media platforms proposing content creators to follow (Linkedin, YouTube)

## Types of recommender systems:
1. Content-Based Filtering : Recommends items based on user's history of likes/rating
2. Collaborative Filtering : Recommends items that user's with similar preferences liked.
3. Hybrid systems: Combine both content-based and collaborative filtering.

### 1. Content Based Filtering

Recommends an item B to user if user liked the item A and item B is similar to item A.

#### Flow:
1. Feature Extraction: A. Combine the item features into a combined string. B. Convert text into numerical features using TFIDF vectorizer.
2. Similarity Computation : Compute cosine similarity between the items.
3. Predict recommendations : Based on similarity matrix, predict recommendations.

In [1]:
import pandas as pd

In [2]:
movies_dict = {
    "movie_id":[200,201,202,203,204],
    "name": ["Movie A", "Movie B","Movie C", "Movie D", "Movie E"],
    "genre":["Action","Romance","Comedy","Action","Comedy"],
    "director": ["Director A", "Director B","Director A", "Director D", "Director B"],
    "actors":['Actor A|Actor B', 'Actor B|Actor C', 'Actor D|Actor E', 'Actor A|Actor F', "Actor A|Actor D"]
}

In [3]:
movies_df = pd.DataFrame (movies_dict)

In [4]:
movies_df

,movie_id,name,genre,director,actors
0,200,Movie A,Action,Director A,Actor A|Actor B
1,201,Movie B,Romance,Director B,Actor B|Actor C
2,202,Movie C,Comedy,Director A,Actor D|Actor E
3,203,Movie D,Action,Director D,Actor A|Actor F
4,204,Movie E,Comedy,Director B,Actor A|Actor D


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [6]:
movies_df["combined_features"] = movies_df["genre"] + " " + movies_df["director"] + " " + movies_df["actors"]

In [7]:
vectorizer=TfidfVectorizer()
tfidf_matrix=vectorizer.fit_transform(movies_df["combined_features"])

In [8]:
for i in range(tfidf_matrix.shape[0]):
    print(tfidf_matrix[i])
    break

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3 stored elements and shape (1, 5)>
  Coords	Values
  (0, 0)	0.6036665474310993
  (0, 3)	0.3565351874675532
  (0, 1)	0.7130703749351064


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
cosine_similarity?

Signature: cosine_similarity(X, Y=None, dense_output=True)
Docstring:
Compute cosine similarity between samples in X and Y.

Cosine similarity, or the cosine kernel, computes similarity as the
normalized dot product of X and Y:

    K(X, Y) = <X, Y> / (||X||*||Y||)

On L2-normalized data, this function is equivalent to linear_kernel.

Read more in the :ref:`User Guide <cosine_similarity>`.

Parameters
----------
X : {array-like, sparse matrix} of shape (n_samples_X, n_features)
    Input data.

Y : {array-like, sparse matrix} of shape (n_samples_Y, n_features),             default=None
    Input data. If ``None``, the output will be the pairwise
    similarities between all samples in ``X``.

dense_output : bool, default=True
    Whether to return dense output even when the input is sparse. If
    ``False``, the output is sparse if both input arrays are sparse.

    .. versionadded:: 0.17
       parameter ``dense_output`` for dense output.

Returns
-------
similarities : ndarray or spa

In [11]:
cosine_sim=cosine_similarity(tfidf_matrix)

In [12]:
cosine_sim

array([[1.        , 0.58131574, 0.6355867 , 1.        , 0.6355867 ],
       [0.58131574, 1.        , 0.58131574, 0.58131574, 0.58131574],
       [0.6355867 , 0.58131574, 1.        , 0.6355867 , 1.        ],
       [1.        , 0.58131574, 0.6355867 , 1.        , 0.6355867 ],
       [0.6355867 , 0.58131574, 1.        , 0.6355867 , 1.        ]])

In [13]:
cosine_sim_df = pd.DataFrame(data= cosine_sim, index= movies_df["name"], columns=movies_df["name"])

In [14]:
cosine_sim_df

name,Movie A,Movie B,Movie C,Movie D,Movie E
name,,,,,
Movie A,1.000000,0.581316,0.635587,1.000000,0.635587
Movie B,0.581316,1.000000,0.581316,0.581316,0.581316
Movie C,0.635587,0.581316,1.000000,0.635587,1.000000
Movie D,1.000000,0.581316,0.635587,1.000000,0.635587
Movie E,0.635587,0.581316,1.000000,0.635587,1.000000


In [15]:
def return_similar_movies(movie_name:str, similarity_df:pd.DataFrame , threshold:float)-> list:
    """Function to return list of similar movies"""
    sim_series = similarity_df.drop(columns=movie_name).loc[movie_name]
    recommended_movies = sim_series[sim_series > threshold].sort_values(ascending=False).index.to_list()
    return recommended_movies

In [16]:
recommended_movies = return_similar_movies(movie_name= "Movie A", similarity_df=cosine_sim_df, threshold=0.6)

In [17]:
recommended_movies

['Movie D', 'Movie C', 'Movie E']

#### Observations: 

Content-Based Filtering focusses on the content of the movie and recommends similar movies to the user based on their previous selection/rating.

## Collaborative Filtering:

Collaborative filtering recommends items based on preferences of similar users.

Relies on : User- Item interactions without needing item features.

#### Flow:
1. Data Preperation: A. Create user-rating matrix. B. Fill nulls where user didn't rate the movie, consider mean, or zero etc.
2. Similarity computation : Compute cosine similarity between the users.
3. Predict user rating.
4. Recommend items to users based on predictions.


In [18]:
ratings_dict = {
    'user_id': [1, 1, 2, 2, 3, 3],
    'movie_id': [101, 102, 101, 103, 102, 104],
    'rating': [5, 3, 4, 2, 5, 4]
}

In [ ]:
ratings_df = pd.DataFrame(ratings_dict)
print(ratings_df)

   user_id  movie_id  rating
0        1       101       5
1        1       102       3
2        2       101       4
3        2       103       2
4        3       102       5
5        3       104       4


In [20]:
user_item_matrix=ratings_df.pivot_table(index="user_id",columns="movie_id",values="rating")

In [ ]:
user_item_matrix

movie_id,101,102,103,104
user_id,,,,
1,5.0,3.0,NaN,NaN
2,4.0,NaN,2.0,NaN
3,NaN,5.0,NaN,4.0


In [22]:
# Cosine similarity between users.
# We can use mean-uzer rating or any other method of filling the nulls, but for now we will use 0.0
user_item_matrix_filled=user_item_matrix.fillna(0.0)
user_similarity = cosine_similarity(user_item_matrix_filled)

In [23]:
user_similarity_df = pd.DataFrame(data= user_similarity, index=user_item_matrix.index, columns= user_item_matrix.index)

In [ ]:
user_similarity_df

user_id,1,2,3
user_id,,,
1,1.000000,0.766965,0.401754
2,0.766965,1.000000,0.000000
3,0.401754,0.000000,1.000000


Now we try to rate the unrated items based on ratings from other users and based on similarity scores.

In [25]:
# Consider a user
user_id = 1

In [26]:
user_ratings = user_item_matrix.loc[user_id]

In [27]:
unrated_items = user_ratings[user_ratings.isna()].index

In [28]:
user_item_matrix

movie_id,101,102,103,104
user_id,,,,
1,5.0,3.0,NaN,NaN
2,4.0,NaN,2.0,NaN
3,NaN,5.0,NaN,4.0


In [29]:
import numpy as np

In [30]:
# Building the prediction dictionary
predictions = {}
for item in unrated_items:
    item_ratings = user_item_matrix[item]
    rated_by = item_ratings[item_ratings.notnull()].index
    similarities = user_similarity_df.loc[user_id, rated_by]
    ratings = item_ratings[rated_by]

    print(f"similarities:{similarities}")

    print(f"ratings:{ratings}")

    weighted_sum = np.dot(similarities, ratings)
    print(f"weighted_sum: {weighted_sum}")

    sum_of_weights = np.sum(similarities)
    print(f"sum_of_weights: {sum_of_weights}")

    predicted_rating = weighted_sum / sum_of_weights if sum_of_weights != 0 else 0
    print(f"predicted_rating:{predicted_rating}")
    predictions[item] = predicted_rating
    print("-"*20)

similarities:user_id
2    0.766965
Name: 1, dtype: float64
ratings:user_id
2    2.0
Name: 103, dtype: float64
weighted_sum: 1.5339299776947406
sum_of_weights: 0.7669649888473703
predicted_rating:2.0
--------------------
similarities:user_id
3    0.401754
Name: 1, dtype: float64
ratings:user_id
3    4.0
Name: 104, dtype: float64
weighted_sum: 1.6070147520167406
sum_of_weights: 0.40175368800418515
predicted_rating:4.0
--------------------


In [31]:
predictions

{103: np.float64(2.0), 104: np.float64(4.0)}

In [32]:
for movie_id, rating in predictions.items():
    print(f"Movie {movie_id}: {rating:.2f}")

Movie 103: 2.00
Movie 104: 4.00
